In [1]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

np.random.seed(42)
random.seed(42)

In [2]:
jobs = [
    {"job_name": "PAY_SETTLEMENT_EOD",   "system": "Payments",             "sla_target_minutes": 90,  "usual_start": "22:00"},
    {"job_name": "PAY_SWIFT_OUT",        "system": "Payments",             "sla_target_minutes": 60,  "usual_start": "23:00"},
    {"job_name": "RECON_CASH_DAILY",     "system": "Reconciliation",       "sla_target_minutes": 120, "usual_start": "01:00"},
    {"job_name": "RECON_NOSTRO",         "system": "Reconciliation",       "sla_target_minutes": 90,  "usual_start": "02:00"},
    {"job_name": "MKTDATA_EOD_LOAD",     "system": "Market Data",          "sla_target_minutes": 60,  "usual_start": "20:00"},
    {"job_name": "MKTDATA_INTRADAY",     "system": "Market Data",          "sla_target_minutes": 30,  "usual_start": "12:00"},
    {"job_name": "RPT_REGULATORY_DAILY", "system": "Regulatory Reporting", "sla_target_minutes": 120, "usual_start": "03:00"},
    {"job_name": "RPT_RISK_EOD",         "system": "Regulatory Reporting", "sla_target_minutes": 90,  "usual_start": "04:00"},
    {"job_name": "SETTLE_EQUITY",        "system": "Settlements",          "sla_target_minutes": 75,  "usual_start": "21:00"},
    {"job_name": "REFDATA_REFRESH",      "system": "Reference Data",       "sla_target_minutes": 45,  "usual_start": "05:00"},
]

jobs_df = pd.DataFrame(jobs)
jobs_df

,job_name,system,sla_target_minutes,usual_start
0,PAY_SETTLEMENT_EOD,Payments,90,22:00
1,PAY_SWIFT_OUT,Payments,60,23:00
2,RECON_CASH_DAILY,Reconciliation,120,01:00
3,RECON_NOSTRO,Reconciliation,90,02:00
4,MKTDATA_EOD_LOAD,Market Data,60,20:00
5,MKTDATA_INTRADAY,Market Data,30,12:00
6,RPT_REGULATORY_DAILY,Regulatory Reporting,120,03:00
7,RPT_RISK_EOD,Regulatory Reporting,90,04:00
8,SETTLE_EQUITY,Settlements,75,21:00
9,REFDATA_REFRESH,Reference Data,45,05:00


In [3]:
error_codes = ["DB_TIMEOUT", "FILE_MISSING", "OUT_OF_MEMORY",
               "UPSTREAM_DELAY", "CONNECTION_REFUSED", "DISK_FULL"]

start_date = datetime(2025, 5, 1)
n_days = 31

rows = []
run_id = 100000

for d in range(n_days):
    run_date = start_date + timedelta(days=d)
    is_month_end = d >= (n_days - 3)

    for _, job in jobs_df.iterrows():
        run_id += 1

        # decide if this run fails
        fail_p = 0.06
        if job["system"] == "Reconciliation":
            fail_p = 0.15
        if job["job_name"] == "RECON_CASH_DAILY":
            fail_p = 0.35
        if is_month_end:
            fail_p = min(fail_p * 2.2, 0.85)
        failed = np.random.random() < fail_p

        # build the timestamps
        hh, mm = map(int, job["usual_start"].split(":"))
        scheduled_start = run_date + timedelta(hours=hh, minutes=mm)
        actual_start = scheduled_start + timedelta(minutes=int(np.random.randint(0, 8)))

        # how long it ran
        sla = job["sla_target_minutes"]
        runtime = sla * 0.65 + np.random.normal(0, sla * 0.15)
        if is_month_end:
            runtime += sla * 0.35
        if failed:
            runtime *= np.random.uniform(0.3, 1.4)
        runtime = max(1, round(runtime))
        actual_end = actual_start + timedelta(minutes=int(runtime))

        # set the status and error code
        if failed:
            status = "Failed"
            if job["job_name"] == "RECON_CASH_DAILY":
                err = np.random.choice(error_codes, p=[0.6, 0.1, 0.1, 0.1, 0.05, 0.05])
            else:
                err = np.random.choice(error_codes)
        else:
            err = None
            status = "Late" if runtime > sla else "Success"

        sla_met = (status == "Success")

        rows.append({
            "run_id": run_id, "job_name": job["job_name"], "system": job["system"],
            "run_date": run_date.date(), "scheduled_start": scheduled_start,
            "actual_start": actual_start, "runtime_minutes": runtime,
            "actual_end": actual_end, "sla_target_minutes": sla,
            "status": status, "sla_met": sla_met, "error_code": err
        })

job_runs = pd.DataFrame(rows)
print("Total runs:", job_runs.shape)
print(job_runs["status"].value_counts())

Total runs: (310, 12)
status
Success    240
Failed      52
Late        18
Name: count, dtype: int64


In [4]:
job_runs.head()

,run_id,job_name,system,run_date,scheduled_start,actual_start,runtime_minutes,actual_end,sla_target_minutes,status,sla_met,error_code
0,100001,PAY_SETTLEMENT_EOD,Payments,2025-05-01,2025-05-01 22:00:00,2025-05-01 22:04:00,66,2025-05-01 23:10:00,90,Success,True,None
1,100002,PAY_SWIFT_OUT,Payments,2025-05-01,2025-05-01 23:00:00,2025-05-01 23:01:00,33,2025-05-01 23:34:00,60,Success,True,None
2,100003,RECON_CASH_DAILY,Reconciliation,2025-05-01,2025-05-01 01:00:00,2025-05-01 01:02:00,19,2025-05-01 01:21:00,120,Failed,False,FILE_MISSING
3,100004,RECON_NOSTRO,Reconciliation,2025-05-01,2025-05-01 02:00:00,2025-05-01 02:07:00,63,2025-05-01 03:10:00,90,Failed,False,DISK_FULL
4,100005,MKTDATA_EOD_LOAD,Market Data,2025-05-01,2025-05-01 20:00:00,2025-05-01 20:03:00,30,2025-05-01 20:33:00,60,Failed,False,UPSTREAM_DELAY


In [5]:
job_runs[job_runs["status"] == "Failed"].head()

,run_id,job_name,system,run_date,scheduled_start,actual_start,runtime_minutes,actual_end,sla_target_minutes,status,sla_met,error_code
2,100003,RECON_CASH_DAILY,Reconciliation,2025-05-01,2025-05-01 01:00:00,2025-05-01 01:02:00,19,2025-05-01 01:21:00,120,Failed,False,FILE_MISSING
3,100004,RECON_NOSTRO,Reconciliation,2025-05-01,2025-05-01 02:00:00,2025-05-01 02:07:00,63,2025-05-01 03:10:00,90,Failed,False,DISK_FULL
4,100005,MKTDATA_EOD_LOAD,Market Data,2025-05-01,2025-05-01 20:00:00,2025-05-01 20:03:00,30,2025-05-01 20:33:00,60,Failed,False,UPSTREAM_DELAY
5,100006,MKTDATA_INTRADAY,Market Data,2025-05-01,2025-05-01 12:00:00,2025-05-01 12:02:00,15,2025-05-01 12:17:00,30,Failed,False,FILE_MISSING
6,100007,RPT_REGULATORY_DAILY,Regulatory Reporting,2025-05-01,2025-05-01 03:00:00,2025-05-01 03:03:00,76,2025-05-01 04:19:00,120,Failed,False,OUT_OF_MEMORY


In [6]:
assignment_map = {
    "Payments": "Payments-L2", "Reconciliation": "Recon-L2",
    "Market Data": "MarketData-L2", "Regulatory Reporting": "AppSupport-L2",
    "Settlements": "AppSupport-L2", "Reference Data": "Unix-L2"
}

rootcause_map = {
    "DB_TIMEOUT": "Database", "OUT_OF_MEMORY": "Capacity", "DISK_FULL": "Capacity",
    "UPSTREAM_DELAY": "Upstream/Feed", "CONNECTION_REFUSED": "Infrastructure",
    "FILE_MISSING": "Application"
}

inc_rows = []
incident_id = 1000
PRB = "PRB0002001"

for _, run in job_runs.iterrows():
    if run["status"] in ("Failed", "Late"):
        incident_id += 1
        inc_id = f"INC{incident_id:07d}"

        # set priority based on how critical the system is
        s = run["system"]
        if s in ("Payments", "Regulatory Reporting"):
            priority = np.random.choice(["P1", "P2", "P3"], p=[0.4, 0.4, 0.2])
        elif s in ("Settlements", "Reconciliation"):
            priority = np.random.choice(["P1", "P2", "P3"], p=[0.15, 0.45, 0.4])
        else:
            priority = np.random.choice(["P2", "P3", "P4"], p=[0.2, 0.4, 0.4])

        # root cause: Late runs are capacity, failures come from the error code
        if run["status"] == "Late":
            root_cause = "Capacity"
        else:
            root_cause = rootcause_map.get(run["error_code"], "Application")

        # when it was raised and how long to fix
        opened_at = run["actual_end"]
        if priority == "P1":
            resolution = int(np.random.randint(15, 90))
        elif priority == "P4":
            resolution = int(np.random.randint(120, 480))
        else:
            resolution = int(np.random.randint(20, 240))

        status_inc = np.random.choice(["Resolved", "Closed", "Open"], p=[0.6, 0.35, 0.05])
        resolved_at = opened_at + timedelta(minutes=resolution) if status_inc != "Open" else None

        # link the recurring recon database timeouts to ONE problem record
        problem_id = PRB if (run["job_name"] == "RECON_CASH_DAILY" and run["error_code"] == "DB_TIMEOUT") else None

        inc_rows.append({
            "incident_id": inc_id, "run_id": run["run_id"], "job_name": run["job_name"],
            "system": run["system"], "priority": priority, "opened_at": opened_at,
            "resolved_at": resolved_at,
            "resolution_minutes": (resolution if status_inc != "Open" else None),
            "assignment_group": assignment_map[run["system"]],
            "root_cause_category": root_cause, "problem_id": problem_id, "status": status_inc
        })

incidents = pd.DataFrame(inc_rows)
print("Total incidents:", incidents.shape)
print(incidents["priority"].value_counts())

Total incidents: (70, 12)
priority
P3    31
P2    21
P1    13
P4     5
Name: count, dtype: int64


In [7]:
incidents[incidents["problem_id"] == "PRB0002001"]

,incident_id,run_id,job_name,system,priority,opened_at,resolved_at,resolution_minutes,assignment_group,root_cause_category,problem_id,status
6,INC0001007,100013,RECON_CASH_DAILY,Reconciliation,P3,2025-05-02 01:56:00,2025-05-02 05:48:00,232.0,Recon-L2,Database,PRB0002001,Resolved
8,INC0001009,100033,RECON_CASH_DAILY,Reconciliation,P3,2025-05-04 01:36:00,2025-05-04 05:08:00,212.0,Recon-L2,Database,PRB0002001,Resolved
10,INC0001011,100053,RECON_CASH_DAILY,Reconciliation,P3,2025-05-06 01:31:00,2025-05-06 02:01:00,30.0,Recon-L2,Database,PRB0002001,Resolved
12,INC0001013,100063,RECON_CASH_DAILY,Reconciliation,P3,2025-05-07 02:27:00,2025-05-07 03:41:00,74.0,Recon-L2,Database,PRB0002001,Closed
13,INC0001014,100073,RECON_CASH_DAILY,Reconciliation,P3,2025-05-08 01:39:00,2025-05-08 03:07:00,88.0,Recon-L2,Database,PRB0002001,Resolved
19,INC0001020,100103,RECON_CASH_DAILY,Reconciliation,P2,2025-05-11 02:20:00,2025-05-11 03:18:00,58.0,Recon-L2,Database,PRB0002001,Resolved
22,INC0001023,100113,RECON_CASH_DAILY,Reconciliation,P1,2025-05-12 02:44:00,2025-05-12 03:47:00,63.0,Recon-L2,Database,PRB0002001,Resolved
26,INC0001027,100123,RECON_CASH_DAILY,Reconciliation,P3,2025-05-13 02:15:00,2025-05-13 06:04:00,229.0,Recon-L2,Database,PRB0002001,Resolved
30,INC0001031,100163,RECON_CASH_DAILY,Reconciliation,P3,2025-05-17 03:09:00,2025-05-17 03:42:00,33.0,Recon-L2,Database,PRB0002001,Resolved
33,INC0001034,100193,RECON_CASH_DAILY,Reconciliation,P1,2025-05-20 01:39:00,2025-05-20 02:11:00,32.0,Recon-L2,Database,PRB0002001,Resolved


In [8]:
# 1. add 5 exact duplicate rows
job_runs_messy = pd.concat([job_runs, job_runs.sample(5, random_state=1)], ignore_index=True)

# 2. wipe the end time on 10 random rows (missing data)
job_runs_messy.loc[job_runs_messy.sample(10, random_state=2).index, "actual_end"] = np.nan

# 3. break the status labels on a few rows (inconsistent text)
job_runs_messy.loc[job_runs_messy[job_runs_messy["status"] == "Success"].sample(5, random_state=3).index, "status"] = "success"
job_runs_messy.loc[job_runs_messy[job_runs_messy["status"] == "Success"].sample(2, random_state=4).index, "status"] = "Sucess"
job_runs_messy.loc[job_runs_messy[job_runs_messy["status"] == "Failed"].sample(4, random_state=5).index, "status"] = "FAILED"

# 4. make 3 rows impossible: end time set before start time
bad = job_runs_messy.sample(3, random_state=6).index
job_runs_messy.loc[bad, "actual_end"] = job_runs_messy.loc[bad, "actual_start"] - pd.Timedelta(minutes=30)

print("Rows now:", job_runs_messy.shape)
print("Duplicate run_ids:", job_runs_messy.duplicated(subset="run_id").sum())

Rows now: (315, 12)
Duplicate run_ids: 5


In [9]:
# make a working copy so we never lose the messy original
runs = job_runs_messy.copy()

# how many duplicate run_ids are there?
print("Duplicates found:", runs.duplicated(subset="run_id").sum())

# remove them, keeping the first copy of each
runs = runs.drop_duplicates(subset="run_id", keep="first")

print("Rows after removing duplicates:", runs.shape)

Duplicates found: 5
Rows after removing duplicates: (310, 12)


In [10]:
# see which columns have missing values
print(runs.isnull().sum())

# create a flag column, empty by default
runs["data_quality_flag"] = ""

# mark the rows where the end time is missing
runs.loc[runs["actual_end"].isnull(), "data_quality_flag"] = "missing_end_time"

print("\nRows flagged as missing end time:", (runs["data_quality_flag"] == "missing_end_time").sum())

run_id                  0
job_name                0
system                  0
run_date                0
scheduled_start         0
actual_start            0
runtime_minutes         0
actual_end             10
sla_target_minutes      0
status                  0
sla_met                 0
error_code            258
dtype: int64

Rows flagged as missing end time: 10


In [11]:
# before: look at the mess
print("BEFORE:")
print(runs["status"].value_counts())

# clean it: trim spaces, fix the casing, then fix the typo
runs["status"] = runs["status"].str.strip().str.title().replace({"Sucess": "Success"})

# after: should be just 3 clean values
print("\nAFTER:")
print(runs["status"].value_counts())

BEFORE:
status
Success    233
Failed      48
Late        18
success      5
FAILED       4
Sucess       2
Name: count, dtype: int64

AFTER:
status
Success    240
Failed      52
Late        18
Name: count, dtype: int64


In [12]:
# make sure both columns are proper datetimes so we can compare them
runs["actual_start"] = pd.to_datetime(runs["actual_start"])
runs["actual_end"] = pd.to_datetime(runs["actual_end"])

# find rows where the end is before the start
invalid = runs[runs["actual_end"] < runs["actual_start"]]
print("Impossible rows (end before start):", len(invalid))

# flag them
runs.loc[runs["actual_end"] < runs["actual_start"], "data_quality_flag"] = "invalid_timestamps"

# check the full tally of everything we've flagged
print("\nData quality flags:")
print(runs["data_quality_flag"].value_counts())

Impossible rows (end before start): 3

Data quality flags:
data_quality_flag
                      297
missing_end_time       10
invalid_timestamps      3
Name: count, dtype: int64


In [13]:
# save the cleaned runs and the incidents to files
runs.to_csv("job_runs_clean.csv", index=False)
incidents.to_csv("incidents.csv", index=False)

print("Files saved:")
print(" - job_runs_clean.csv  (", runs.shape[0], "rows )")
print(" - incidents.csv  (", incidents.shape[0], "rows )")

Files saved:
 - job_runs_clean.csv  ( 310 rows )
 - incidents.csv  ( 70 rows )


In [14]:
import sqlite3

# create a database that lives in memory
conn = sqlite3.connect(":memory:")

# write both tables into it
runs.to_sql("job_runs", conn, index=False, if_exists="replace")
incidents.to_sql("incidents", conn, index=False, if_exists="replace")

# quick test: count the rows
import pandas as pd
print(pd.read_sql("SELECT COUNT(*) AS total FROM job_runs", conn))

   total
0    310


In [15]:
# overall success rate
q1 = """
SELECT ROUND(100.0 * SUM(CASE WHEN status='Success' THEN 1 ELSE 0 END) / COUNT(*), 1) AS success_rate_pct
FROM job_runs
"""
print(pd.read_sql(q1, conn))

# failure rate broken down by system
q2 = """
SELECT system,
       COUNT(*) AS total_runs,
       SUM(CASE WHEN status='Failed' THEN 1 ELSE 0 END) AS failures,
       ROUND(100.0 * SUM(CASE WHEN status='Failed' THEN 1 ELSE 0 END) / COUNT(*), 1) AS failure_pct
FROM job_runs
GROUP BY system
ORDER BY failure_pct DESC
"""
print(pd.read_sql(q2, conn))

   success_rate_pct
0              77.4
                 system  total_runs  failures  failure_pct
0        Reconciliation          62        26         41.9
1           Market Data          62        12         19.4
2  Regulatory Reporting          62         8         12.9
3              Payments          62         4          6.5
4           Settlements          31         1          3.2
5        Reference Data          31         1          3.2


In [16]:
# SLA met % by system (lower = worse)
q3 = """
SELECT system,
       ROUND(100.0 * SUM(sla_met) / COUNT(*), 1) AS sla_met_pct
FROM job_runs
GROUP BY system
ORDER BY sla_met_pct ASC
"""
print(pd.read_sql(q3, conn))

# which individual jobs fail the most (only show repeat offenders)
q4 = """
SELECT job_name, COUNT(*) AS failures
FROM job_runs
WHERE status='Failed'
GROUP BY job_name
HAVING COUNT(*) > 3
ORDER BY failures DESC
"""
print(pd.read_sql(q4, conn))

                 system  sla_met_pct
0        Reconciliation         53.2
1           Market Data         74.2
2  Regulatory Reporting         79.0
3              Payments         88.7
4        Reference Data         90.3
5           Settlements         93.5
               job_name  failures
0      RECON_CASH_DAILY        18
1          RECON_NOSTRO         8
2      MKTDATA_EOD_LOAD         8
3          RPT_RISK_EOD         4
4  RPT_REGULATORY_DAILY         4
5         PAY_SWIFT_OUT         4
6      MKTDATA_INTRADAY         4


In [17]:
# JOIN the two tables: failed runs alongside their incident details
q5 = """
SELECT r.run_id, r.job_name, r.run_date, r.status,
       i.incident_id, i.priority, i.root_cause_category
FROM job_runs r
JOIN incidents i ON r.run_id = i.run_id
ORDER BY r.run_date
LIMIT 10
"""
print(pd.read_sql(q5, conn))

# average time to resolve, by priority (this is MTTR)
q6 = """
SELECT priority,
       COUNT(*) AS incidents,
       ROUND(AVG(resolution_minutes), 0) AS avg_mttr_min
FROM incidents
WHERE resolution_minutes IS NOT NULL
GROUP BY priority
ORDER BY priority
"""
print(pd.read_sql(q6, conn))

   run_id              job_name    run_date  status incident_id priority  \
0  100003      RECON_CASH_DAILY  2025-05-01  Failed  INC0001001       P1   
1  100004          RECON_NOSTRO  2025-05-01  Failed  INC0001002       P2   
2  100005      MKTDATA_EOD_LOAD  2025-05-01  Failed  INC0001003       P4   
3  100006      MKTDATA_INTRADAY  2025-05-01  Failed  INC0001004       P2   
4  100007  RPT_REGULATORY_DAILY  2025-05-01  Failed  INC0001005       P2   
5  100012         PAY_SWIFT_OUT  2025-05-02  Failed  INC0001006       P2   
6  100013      RECON_CASH_DAILY  2025-05-02  Failed  INC0001007       P3   
7  100024          RECON_NOSTRO  2025-05-03  Failed  INC0001008       P2   
8  100033      RECON_CASH_DAILY  2025-05-04  Failed  INC0001009       P3   
9  100052         PAY_SWIFT_OUT  2025-05-06  Failed  INC0001010       P2   

  root_cause_category  
0         Application  
1            Capacity  
2       Upstream/Feed  
3         Application  
4            Capacity  
5       Upstream/Fe

In [18]:
# why does RECON_CASH_DAILY keep failing? group its failures by root cause and error
q7 = """
SELECT r.job_name, i.root_cause_category, r.error_code, COUNT(*) AS occurrences
FROM job_runs r
JOIN incidents i ON r.run_id = i.run_id
WHERE r.job_name = 'RECON_CASH_DAILY'
GROUP BY i.root_cause_category, r.error_code
ORDER BY occurrences DESC
"""
print(pd.read_sql(q7, conn))

# how many incidents are linked to the one problem record?
q8 = """
SELECT problem_id, COUNT(*) AS linked_incidents
FROM incidents
WHERE problem_id IS NOT NULL
GROUP BY problem_id
"""
print(pd.read_sql(q8, conn))

           job_name root_cause_category          error_code  occurrences
0  RECON_CASH_DAILY            Database          DB_TIMEOUT           14
1  RECON_CASH_DAILY         Application        FILE_MISSING            1
2  RECON_CASH_DAILY            Capacity           DISK_FULL            1
3  RECON_CASH_DAILY      Infrastructure  CONNECTION_REFUSED            1
4  RECON_CASH_DAILY       Upstream/Feed      UPSTREAM_DELAY            1
   problem_id  linked_incidents
0  PRB0002001                14


In [19]:
# incident count by root cause (what's hurting us most)
q9 = """
SELECT root_cause_category, COUNT(*) AS incidents
FROM incidents
GROUP BY root_cause_category
ORDER BY incidents DESC
"""
print(pd.read_sql(q9, conn))

# the serious incidents (P1 and P2)
q10 = """
SELECT priority, COUNT(*) AS incidents
FROM incidents
WHERE priority IN ('P1', 'P2')
GROUP BY priority
"""
print(pd.read_sql(q10, conn))

# failure trend: failures per day (look for the month-end spike)
q11 = """
SELECT run_date, COUNT(*) AS failures
FROM job_runs
WHERE status = 'Failed'
GROUP BY run_date
ORDER BY run_date
"""
print(pd.read_sql(q11, conn).tail(8))

  root_cause_category  incidents
0            Capacity         34
1            Database         19
2       Upstream/Feed          8
3         Application          5
4      Infrastructure          4
  priority  incidents
0       P1         13
1       P2         21
      run_date  failures
18  2025-05-22         2
19  2025-05-23         3
20  2025-05-26         2
21  2025-05-27         1
22  2025-05-28         1
23  2025-05-29         3
24  2025-05-30         4
25  2025-05-31         2
